# Gaussian Blur + Edge Detection Pipeline — AXI Stream
Sends each video frame through the HLS Gaussian blur IP, then the HLS edge detection IP,
both connected via AXI Stream on the FPGA. The PS drives the pipeline using AXI DMA.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import cv2
import os
import time

## 1. Configuration

In [ ]:
# --- Paths ---
BITSTREAM   = "update3.bit"   # .bit + .hwh must share the same base name
INPUT_VIDEO = "input.mp4"
OUTPUT_DIR  = "output_frames"

# --- Frame dimensions — must match IMG_HEIGHT/IMG_WIDTH in HLS headers ---
IMG_HEIGHT = 480
IMG_WIDTH  = 640

# --- AXI-Lite control register offsets ---
# Verify against xgaussian_blur_hw.h and xedge_detect_hw.h in your Vitis HLS solution
AP_CTRL = 0x00   # bit 0 = ap_start, bit 1 = ap_done, bit 2 = ap_idle

# gaussian_blur scalar args: height, width
GB_HEIGHT = 0x10
GB_WIDTH  = 0x18

# edge_detect scalar args: height, width, threshold
ED_HEIGHT    = 0x10
ED_WIDTH     = 0x18
ED_THRESHOLD = 0x20

# Edge detection sensitivity — lower than raw-frame value since Gaussian pre-blurs
THRESHOLD = 20

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load Overlay

In [ ]:
print("Loading overlay...")
ol       = Overlay(BITSTREAM)
dma      = ol.axi_dma_0
gaussian = ol.gaussian_blur_0
edge     = ol.edge_detect_0
print("Overlay loaded.")
print(f"  gaussian_blur_0 : {gaussian}")
print(f"  edge_detect_0   : {edge}")
print(f"  axi_dma_0       : {dma}")

## 3. Allocate DMA Buffers

In [ ]:
in_buf  = allocate(shape=(IMG_HEIGHT * IMG_WIDTH,), dtype=np.uint8)
out_buf = allocate(shape=(IMG_HEIGHT * IMG_WIDTH,), dtype=np.uint8)

print(f"Input  buffer physical address: 0x{in_buf.physical_address:08X}")
print(f"Output buffer physical address: 0x{out_buf.physical_address:08X}")

## 4. Helper — Run One Frame Through the Pipeline

In [ ]:
def run_pipeline(gray_frame: np.ndarray) -> np.ndarray:
    """
    Send a single grayscale frame through the HLS pipeline:
        DMA -> gaussian_blur -> edge_detect -> DMA
    Returns a binary edge map (0 or 255) shaped (IMG_HEIGHT, IMG_WIDTH).
    """
    # Copy flattened frame into contiguous input buffer and flush PS cache
    in_buf[:] = gray_frame.flatten()
    in_buf.flush()

    # Configure gaussian_blur AXI-Lite registers
    gaussian.write(GB_HEIGHT, IMG_HEIGHT)
    gaussian.write(GB_WIDTH,  IMG_WIDTH)

    # Configure edge_detect AXI-Lite registers
    edge.write(ED_HEIGHT,    IMG_HEIGHT)
    edge.write(ED_WIDTH,     IMG_WIDTH)
    edge.write(ED_THRESHOLD, THRESHOLD)

    # Start both IPs — they block waiting for stream data
    # Start edge_detect first so it is ready when gaussian_blur sends data
    edge.write(AP_CTRL,     0x01)
    gaussian.write(AP_CTRL, 0x01)

    # Arm DMA receive channel before sending so no output beats are dropped
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.transfer(in_buf)

    # Block until both DMA channels complete
    dma.sendchannel.wait()
    dma.recvchannel.wait()

    # Invalidate PS cache so we read fresh DDR data written by the PL
    out_buf.invalidate()

    return np.array(out_buf).reshape(IMG_HEIGHT, IMG_WIDTH)

## 5. Process Video

In [ ]:
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {INPUT_VIDEO}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
print(f"Video: {total_frames} frames @ {fps:.1f} fps")

frame_idx = 0
times     = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Resize to match HLS block dimensions if needed
    if frame.shape[0] != IMG_HEIGHT or frame.shape[1] != IMG_WIDTH:
        frame = cv2.resize(frame, (IMG_WIDTH, IMG_HEIGHT))

    # Convert to grayscale — HLS pipeline is single-channel
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Run through HLS Gaussian blur -> edge detect pipeline
    t0       = time.perf_counter()
    edge_map = run_pipeline(gray)
    times.append(time.perf_counter() - t0)

    # Overlay red edges on the original colour frame
    output = frame.copy()
    output[edge_map == 255] = [0, 0, 255]

    out_path = os.path.join(OUTPUT_DIR, f"frame_{frame_idx:05d}.png")
    cv2.imwrite(out_path, output)

    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"  Processed {frame_idx}/{total_frames} frames")

cap.release()

avg_ms = (sum(times) / len(times)) * 1000 if times else 0
print(f"\nDone. {frame_idx} frames saved to '{OUTPUT_DIR}/'")
print(f"Avg pipeline time per frame: {avg_ms:.2f} ms")

## 6. (Optional) Reassemble Frames into a Video

In [ ]:
OUTPUT_VIDEO = "output_edges.avi"

fourcc  = cv2.VideoWriter_fourcc(*'XVID')
writer  = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (IMG_WIDTH, IMG_HEIGHT), isColor=True)

frame_files = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith(".png"))
for fname in frame_files:
    img = cv2.imread(os.path.join(OUTPUT_DIR, fname))
    writer.write(img)

writer.release()
print(f"Saved reassembled video: {OUTPUT_VIDEO}")

## 7. Free Buffers

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()
print("Buffers freed.")